In [5]:
import os
import time
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt

In [6]:
#配置 
data_root = "./data1"  
batch_size = 64
num_epochs = 10
lr = 0.001
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


数据预处理 

In [8]:
#训练增强
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),#裁剪到244*244格式
    transforms.RandomHorizontalFlip(),#随机水平翻转
    transforms.ToTensor(),#转为张量
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])#采用ImageNet通用均值和标准差
])
#验证只用基本变换
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
#加载数据（ImageFolder：根据文件夹自动生成分类标签）
train_dataset = datasets.ImageFolder(root=os.path.join(data_root, "train"), transform=train_transform)
val_dataset   = datasets.ImageFolder(root=os.path.join(data_root, "val"), transform=val_transform)
#打包分批
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

num_classes = len(train_dataset.classes)
print(f"类别数：{num_classes}")
print(f"类别：{train_dataset.classes}")


类别数：30
类别：['aloevera', 'banana', 'bilimbi', 'cantaloupe', 'cassava', 'coconut', 'corn', 'cucumber', 'curcuma', 'eggplant', 'galangal', 'ginger', 'guava', 'kale', 'longbeans', 'mango', 'melon', 'orange', 'paddy', 'papaya', 'peperchili', 'pineapple', 'pomelo', 'shallot', 'soybeans', 'spinach', 'sweetpotatoes', 'tobacco', 'waterapple', 'watermelon']


In [9]:
#可配置的 ResNet
class BasicBlock(nn.Module):
    expansion = 1 #ResNet固定参数
    def __init__(self, in_planes, planes, stride=1, use_bn=True, use_res=True):
        super().__init__()
        #全局开关
        self.use_bn = use_bn  #是否启用批量归一化
        self.use_res = use_res #是否启用残差链接

        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=not use_bn)
        self.bn1 = nn.BatchNorm2d(planes) if use_bn else nn.Identity()
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=not use_bn)
        self.bn2 = nn.BatchNorm2d(planes) if use_bn else nn.Identity()
        #输入x维度与输出不一致时，用1*1卷积对齐尺寸
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes) if use_bn else nn.Identity()
            )
    #向前传播forward
    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.use_res:
            out += self.shortcut(x)
        out = torch.relu(out)
        return out

ResNet类

In [11]:
class ResNet(nn.Module):
    def __init__(self, block, num_blocks, num_classes=1000, use_bn=True, use_res=True):
        super().__init__()
        self.in_planes = 64
        self.use_bn = use_bn
        self.use_res = use_res
        #网络架构
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=not use_bn)
        self.bn1 = nn.BatchNorm2d(64) if use_bn else nn.Identity()
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)
    #残差层的构造函数
    def _make_layer(self, block, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_planes, planes, stride, self.use_bn, self.use_res))
            self.in_planes = planes * block.expansion
        return nn.Sequential(*layers)

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = torch.flatten(out, 1)
        out = self.fc(out)
        return out
#快捷构建函数
def resnet18(num_classes=num_classes, use_bn=True, use_res=True):
    return ResNet(BasicBlock, [2,2,2,2], num_classes, use_bn, use_res)

训练和验证函数

In [ ]:
#单轮训练函数
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)#计算交叉熵损失
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * inputs.size(0)
        _, pred = outputs.max(1)#取出预测概率最大的索引
        correct += pred.eq(labels).sum().item()
        total += labels.size(0)
    avg_loss = total_loss / total
    acc = 100. * correct / total
    return avg_loss, acc
#测试集评估函数
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            total_loss += loss.item() * inputs.size(0)
            _, pred = outputs.max(1)
            correct += pred.eq(labels).sum().item()
            total += labels.size(0)
    avg_loss = total_loss / total
    acc = 100. * correct / total
    return avg_loss, acc
#整体训练函数
def train_model(model, name, epochs=num_epochs):
    print(f"\n===== 开始训练：{name} =====")
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    best_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(1, epochs+1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        #指标存入历史字典
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        #打印结果
        print(f"Epoch {epoch:2d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} Acc: {val_acc:.2f}%")
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), f"{name}_best.pth")
    print(f"{name} 最佳验证准确率: {best_acc:.2f}%")
    return history, best_acc

运行三个实验 

In [ ]:
results = {}
#标准ResNet
model_std = resnet18(use_bn=True, use_res=True)
hist_std, acc_std = train_model(model_std, "ResNet_standard")

#去掉 BN
model_no_bn = resnet18(use_bn=False, use_res=True)
hist_no_bn, acc_no_bn = train_model(model_no_bn, "ResNet_no_BN")

#去掉残差
model_no_res = resnet18(use_bn=True, use_res=False)
hist_no_res, acc_no_res = train_model(model_no_res, "ResNet_no_residual")


===== 开始训练：ResNet_standard =====


绘制损失和准确率曲线

In [ ]:
def plot_history(histories, title):
    plt.figure(figsize=(12,4))
    for name, hist in histories.items():
        plt.subplot(1,2,1)
        plt.plot(hist['val_acc'], label=name)
        plt.title('Validation Accuracy')
        plt.xlabel('Epoch')
        plt.ylabel('Acc (%)')
        plt.legend()
        plt.subplot(1,2,2)
        plt.plot(hist['val_loss'], label=name)
        plt.title('Validation Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(f"{title}.png")
    plt.show()

all_hist = {
    "Standard": hist_std,
    "No BN": hist_no_bn,
    "No Residual": hist_no_res
}
plot_history(all_hist, "Comparison_ResNet_Experiments")

样本预测

In [ ]:
def show_predictions(model, val_loader, device, num_images=6):
    model.eval()
    images, labels, preds = [], [], []
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            images.extend(inputs.cpu())
            labels.extend(targets)
            preds.extend(predicted.cpu())
            if len(images) >= num_images:
                break
    # 反归一化
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    plt.figure(figsize=(12, 8))
    for i in range(num_images):
        plt.subplot(2, 3, i+1)
        img = images[i] * std + mean  # 反归一化
        img = img.clamp(0, 1).permute(1,2,0)
        plt.imshow(img)
        true_label = val_dataset.classes[labels[i]]
        pred_label = val_dataset.classes[preds[i]]
        color = 'green' if labels[i] == preds[i] else 'red'
        plt.title(f'True: {true_label}\nPred: {pred_label}', color=color, fontsize=10)
        plt.axis('off')
    plt.tight_layout()
    plt.savefig('sample_predictions.png')
    plt.show()

# 调用（以标准模型为例）
show_predictions(model_std, val_loader, device, num_images=6)